# 🖥️ Which RunPod GPU should I use?

This notebook loads **Mistral-7B-Instruct-v0.3** (~7.2 B params). The VRAM you need depends on the **precision** you pick in the selector below — choose a pod that clears the *"1 model"* column:

| Precision | Weights | ~VRAM (1 model) | Recommended RunPod GPU | ~Price / hr |
|:---|:---|:---|:---|:---|
| **4-bit** (NF4) | ~3.6 GB | ~6 GB | **RTX A5000 · 24 GB** | **\$0.27** |
| **16-bit** (bf16) | ~14.5 GB | ~16–18 GB | **RTX A5000 · 24 GB** | **\$0.27** |
| **32-bit** (fp32) | ~29 GB | ~30–32 GB | **RTX A6000 · 48 GB** (or A40 48 GB) | **\$0.49** (A40 \$0.44) |

_Prices: RunPod Community Cloud, on-demand, ~July 2026 — they change often, check [runpod.io/pricing](https://www.runpod.io/pricing). Also set the pod's **Container/Volume disk to ≥ 100 GB** (the model is downloaded and saved several times)._

**⚠️ Running all four sections top-to-bottom keeps every loaded model in VRAM (up to 4 copies):**

| Precision | All 4 models | GPU to run the whole notebook in one go |
|:---|:---|:---|
| 4-bit | ~18 GB | RTX A5000 24 GB ✅ (what this course uses) |
| 16-bit | ~60 GB | A100 80 GB (~\$1.39 / hr) |
| 32-bit | ~120 GB | not practical — run one section at a time |

To stay on the cheap 24 GB card for 16-/32-bit, free the previous model before loading the next:

```python
del base_model            # or model_hub_loaded_from_hub / model_loaded / model_loaded_fromHF
import gc, torch; gc.collect(); torch.cuda.empty_cache()
```


# Install libraries

In [1]:
# Fix torchaudio/torch CUDA version mismatch, then pin transformers to a stable 4.x release
%pip install --upgrade --force-reinstall torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install "transformers==4.45.2" --force-reinstall
%pip install bitsandbytes
%pip install accelerate
%pip install peft
%pip install datasets
%pip install evaluate
%pip install trl
%pip install matplotlib
%pip install tensorboard
%pip install sentencepiece


Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp311-cp311-linux_x86_64.whl (3.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp311-cp311-linux_x86_64.whl (780.5 MB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached https://download.pytorch.org/whl/c

In [2]:
import torch
for i in range(torch.cuda.device_count()):
   print(torch.cuda.get_device_properties(i).name)


NVIDIA L4


In [13]:
from huggingface_hub import login

API_TOKEN = "HF_TOKEN_REMOVED"
login(token=API_TOKEN)


# ⚙️ Choose precision -- run ONE of the three cells below

Run **exactly one** of the 4-bit / 16-bit / 32-bit cells. It sets `PRECISION` and `MODEL_KWARGS`,
which every section below uses (model loading, the on-disk folder `SAVED_MODEL_<precision>`, and the
Hub repo names `my-base-mistral-<precision>-...`). To switch precision later, run a different cell,
then re-run the **Load the model** cell.

In [4]:
# --- 4-bit (bitsandbytes NF4) ---
import torch
from transformers import BitsAndBytesConfig

PRECISION = "4bit"
MODEL_KWARGS = dict(
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    ),
    dtype=torch.bfloat16,
    device_map="cuda",
)
print("Selected precision:", PRECISION)


Selected precision: 4bit


In [ ]:
# --- 16-bit (half precision, bfloat16) ---
import torch

PRECISION = "16bit"
MODEL_KWARGS = dict(
    dtype=torch.bfloat16,
    device_map="cuda",
)
print("Selected precision:", PRECISION)


In [ ]:
# --- 32-bit (full precision, float32) ---
import torch

PRECISION = "32bit"
MODEL_KWARGS = dict(
    dtype=torch.float32,
    device_map="cuda",
)
print("Selected precision:", PRECISION)


# Base model from HUB

**What this section demonstrates:** the starting point. We download the original `mistralai/Mistral-7B-Instruct-v0.3` from the Hugging Face Hub at the precision you selected, verify it generates correctly, then **persist it two ways** — save a copy to local disk *and* push a copy to your own Hub repo. Those copies feed the next three sections.

```mermaid
flowchart LR
    ORIG["🤗 Original Hub<br/>Mistral-7B-Instruct-v0.3"] -->|"① load"| BM["base_model"]
    BM -->|"① save"| DISK["💾 disk<br/>SAVED_MODEL_&lt;prec&gt;"]
    BM -->|"① push"| HUB1["📦 Your Hub<br/>…-from-hub"]
    DISK -->|"③ load"| ML["model_loaded"]
    ML -->|"③ push"| HUB2["📦 Your Hub<br/>…-from-disk"]
    HUB1 -->|"② load + test"| OK2["✅ works"]
    HUB2 -->|"④ load + test"| OK4["✅ works"]
    style ORIG fill:#2e7d32,color:#fff,stroke:#1b5e20
```


### Load the model

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Silence transformers' deprecation notice about `torch_dtype` in the model's
# own config.json (upstream in the Mistral repo; harmless).
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

# Names derived from the precision you selected above -- used by every section below.
SAVED_DIR      = f"SAVED_MODEL_{PRECISION}"
REPO_FROM_HUB  = f"habert75/my-base-mistral-{PRECISION}-from-hub"
REPO_FROM_DISK = f"habert75/my-base-mistral-{PRECISION}-from-disk"

base_model = AutoModelForCausalLM.from_pretrained(model_name, **MODEL_KWARGS)
base_tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Loaded {model_name}  |  precision={PRECISION}  |  hub repo={REPO_FROM_HUB}")


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Loaded mistralai/Mistral-7B-Instruct-v0.3  |  precision=4bit  |  hub repo=habert75/my-base-mistral-4bit-from-hub


Log model and tokenizer

In [8]:
def describe(model, tokenizer):
    """Print key facts about a loaded model + tokenizer. Reused in every section below."""
    print("--- Model ---")
    print("Type:", type(model))
    print("Architecture:", model)
    print("Vocab size:", model.config.vocab_size)
    print("Model configuration:", model.config)
    print("Input embeddings:", model.get_input_embeddings())
    print("Output embeddings:", model.get_output_embeddings())
    print("--- Tokenizer ---")
    print("Type:", type(tokenizer))
    print("Special tokens:", tokenizer.special_tokens_map)
    print("All tokens count:", len(tokenizer))
    print("Padding side:", tokenizer.padding_side)


describe(base_model, base_tokenizer)


--- Model ---
Type: <class 'transformers.models.mistral.modeling_mistral.MistralForCausalLM'>
Architecture: MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((40

### Test the model

#### via Pipeline

In [9]:
from transformers import pipeline

PROMPT = "What is AutoGen in abstract?"


def test_pipeline(model, tokenizer, prompt=PROMPT):
    """Test a model with a text-generation pipeline. Reused in every section below."""
    my_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

    # Mistral-7B-Instruct is instruction-tuned, so we pass a chat (list of messages)
    # instead of a raw string. The pipeline then auto-applies the tokenizer's chat
    # template, giving a coherent answer (a raw string would ignore the template and ramble).
    messages = [{"role": "user", "content": prompt}]
    result = my_pipeline(messages, max_new_tokens=256, return_full_text=False)
    print(result[0]["generated_text"])


test_pipeline(base_model, base_tokenizer)


AutoGen is a programming tool that automatically generates source code based on high-level specifications. It is often used for creating large, complex programs or systems, especially in areas like compiler construction, network protocols, and data processing pipelines.

The user provides a set of templates and rules, and AutoGen generates code that conforms to those specifications. This can significantly reduce the amount of manual coding required, making the development process more efficient. It can also help ensure consistency and maintainability of the generated code, as the same rules are applied throughout the codebase.

AutoGen is not a specific tool but a general concept. There are several implementations of AutoGen-like tools, such as Coco/R, SDF, and Yacc/Bison, each with its own specific features and capabilities.


#### via Model and Tokenizer

In [10]:
def test_generate(model, tokenizer, prompt=PROMPT):
    """Test a model by manually applying the chat template + generate. Reused below."""
    # Instruction-tuned model -> build inputs from a chat via the tokenizer's chat template.
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True
    ).to("cuda")

    # Generate, then decode only the answer.
    result = model.generate(**model_inputs, max_new_tokens=256)
    print(tokenizer.decode(result[0], skip_special_tokens=True))


test_generate(base_model, base_tokenizer)


What is AutoGen in abstract? AutoGen is a programming system that automates the generation of source code from a high-level description. It was developed by David A. Bayer and Stephen C. Johnson in the early 1980s at Bell Labs.

The system works by defining a set of templates that describe the structure of the desired code, along with functions that generate the actual code based on input parameters. These templates can be written in a language similar to C, but with additional features for specifying the structure of the output code.

AutoGen is often used to generate code for complex systems, such as operating systems, network protocols, and database systems. It can help reduce the amount of manual coding required, improve code consistency, and make it easier to modify and maintain large codebases.

AutoGen is not a widely used system today, but its principles have influenced other code generation tools, such as ANTLR, Jison, and Bison.


#### via Tool call (single turn)

In [ ]:
# Single-turn tool calling: pass `tools` to the chat template. An instruction-tuned
# model should answer with a tool call (function name + JSON arguments) instead of prose.
STOCK_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current price of a stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The ticker symbol for the stock, e.g. GOOG",
                    }
                },
                "required": ["ticker"],
            },
        },
    }
]


def test_tool_call(model, tokenizer, prompt, tools, max_new_tokens=256):
    """Single-turn tool calling: pass `tools`; the model should emit a tool call. Reused below."""
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)

    # Greedy decoding -> deterministic tool call. Keep special tokens so the
    # tool-call markers (e.g. [TOOL_CALLS]) stay visible.
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_call(base_model, base_tokenizer, "What is the current price of MSFT?", STOCK_TOOLS)


#### via Conversation with tool calls

In [ ]:
def test_tool_conversation(model, tokenizer, max_new_tokens=256):
    """Multi-turn conversation that already contains an assistant tool call + a tool
    response, then a fresh user turn. Confirms the chat template round-trips the
    `tool_calls` / `tool` roles, then answers the latest turn. Reused below."""
    conversation = [
        {"role": "user", "content": "What's the weather like in Paris?"},
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "id": "call12345",
                    "type": "function",
                    "function": {
                        "name": "get_current_weather",
                        "arguments": '{"location": "Paris, France", "format": "celsius"}',
                    },
                }
            ],
        },
        {"role": "tool", "content": '{"content": 22}', "tool_call_id": "call12345"},
        {
            "role": "assistant",
            "content": "The current temperature in Paris, France is 22 degrees Celsius.",
        },
        {"role": "user", "content": "What's the weather like in San Francisco right now?"},
    ]

    weather_tools = [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        },
                        "format": {
                            "type": "string",
                            "enum": ["celsius", "fahrenheit"],
                            "description": "The temperature unit to use. Infer this from the users location.",
                        },
                    },
                    "required": ["location", "format"],
                },
            },
        }
    ]

    model_inputs = tokenizer.apply_chat_template(
        conversation, tools=weather_tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    print("Tokenized conversation input_ids:")
    print(model_inputs["input_ids"])

    # Answer the latest user turn (should be another get_current_weather tool call for SF).
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print("\nModel response:")
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_conversation(base_model, base_tokenizer)


### Save the model (to disk)

In [11]:
base_model.save_pretrained(SAVED_DIR)

base_tokenizer.save_pretrained(SAVED_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('SAVED_MODEL_4bit/tokenizer_config.json',
 'SAVED_MODEL_4bit/chat_template.jinja',
 'SAVED_MODEL_4bit/tokenizer.json')

### Push the model (to HUB)

In [14]:
# push_to_hub() -> save_pretrained(save_original_format=True) can raise NotImplementedError
# in transformers 5.x, so save with save_original_format=False and upload the folder instead.
from huggingface_hub import upload_folder, create_repo

tmp = f"PUSH_TMP_{PRECISION}_from_hub"
create_repo(REPO_FROM_HUB, token=API_TOKEN, exist_ok=True)
base_model.save_pretrained(tmp, save_original_format=False)
base_tokenizer.save_pretrained(tmp)
upload_folder(repo_id=REPO_FROM_HUB, folder_path=tmp, token=API_TOKEN)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/habert75/my-base-mistral-4bit-from-hub/commit/2d6e91c5d99ac080e1f989011f962dfa5425acb9', commit_message='Upload folder using huggingface_hub', commit_description='', oid='2d6e91c5d99ac080e1f989011f962dfa5425acb9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/habert75/my-base-mistral-4bit-from-hub', endpoint='https://huggingface.co', repo_type='model', repo_id='habert75/my-base-mistral-4bit-from-hub'), pr_revision=None, pr_num=None)

# Hub model from HUB

**What this section demonstrates:** a full round-trip through the Hub. We reload the model we just pushed (from *your* repo `my-base-mistral-<precision>-from-hub`) and test it. The point: a model downloaded back from the Hub behaves exactly like the original — pushing to the Hub is lossless.

```mermaid
flowchart LR
    ORIG["🤗 Original Hub<br/>Mistral-7B-Instruct-v0.3"] -->|"① load"| BM["base_model"]
    BM -->|"① save"| DISK["💾 disk<br/>SAVED_MODEL_&lt;prec&gt;"]
    BM -->|"① push"| HUB1["📦 Your Hub<br/>…-from-hub"]
    DISK -->|"③ load"| ML["model_loaded"]
    ML -->|"③ push"| HUB2["📦 Your Hub<br/>…-from-disk"]
    HUB1 -->|"② load + test"| OK2["✅ works"]
    HUB2 -->|"④ load + test"| OK4["✅ works"]
    style HUB1 fill:#2e7d32,color:#fff,stroke:#1b5e20
    style OK2 fill:#2e7d32,color:#fff,stroke:#1b5e20
```


### Load the model

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_hub_loaded_from_hub = AutoModelForCausalLM.from_pretrained(
    REPO_FROM_HUB, device_map="auto"
)
tokenizer_hub_loaded_from_hub = AutoTokenizer.from_pretrained(REPO_FROM_HUB)


config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.14GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.67M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.96k [00:00<?, ?B/s]

Log model and tokenizer

In [16]:
describe(model_hub_loaded_from_hub, tokenizer_hub_loaded_from_hub)


--- Model ---
Type: <class 'transformers.models.mistral.modeling_mistral.MistralForCausalLM'>
Architecture: MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((40

### Test the model

#### via Pipeline

In [17]:
test_pipeline(model_hub_loaded_from_hub, tokenizer_hub_loaded_from_hub)


AutoGen is a program that generates C or C++ source code based on a high-level description of the desired functionality. It is often used for creating code for data structures, parsers, and other components where the structure is complex or repetitive. AutoGen allows developers to write rules or templates that describe the desired output, and the tool generates the code according to these rules. This can save time and reduce errors in the coding process, especially for large and complex programs. AutoGen is often used in conjunction with lex and yacc (or their modern equivalents flex and bison) for generating parser code.


#### via Model and Tokenizer

In [18]:
test_generate(model_hub_loaded_from_hub, tokenizer_hub_loaded_from_hub)


What is AutoGen in abstract? AutoGen is a programming system that automates the generation of source code from a high-level description. It was developed by David A. Bayer and Stephen C. Johnson in the early 1980s at Bell Labs.

The system works by defining a set of templates that describe the structure of the desired code, along with functions that generate the actual code based on input parameters. These templates can be written in a language similar to C, but with additional features for specifying the structure of the output code.

AutoGen is often used to generate code for complex systems, such as operating systems, network protocols, and database systems. It can help reduce the amount of manual coding required, improve code consistency, and make it easier to modify and maintain large codebases.

AutoGen is not a widely used system today, but its principles have influenced other code generation tools, such as ANTLR, Jison, and Bison.


#### via Tool call (single turn)

In [ ]:
test_tool_call(model_hub_loaded_from_hub, tokenizer_hub_loaded_from_hub, "What is the current price of MSFT?", STOCK_TOOLS)


#### via Conversation with tool calls

In [19]:
test_tool_conversation(model_hub_loaded_from_hub, tokenizer_hub_loaded_from_hub)


NameError: name 'test_tool_conversation' is not defined

# Base model from DISK

**What this section demonstrates:** loading from local storage instead of the network. We reload the model from the on-disk folder `SAVED_MODEL_<precision>` (saved in the first section), test it, then push this disk copy to a second Hub repo. Confirms a locally-saved model is fully functional.

```mermaid
flowchart LR
    ORIG["🤗 Original Hub<br/>Mistral-7B-Instruct-v0.3"] -->|"① load"| BM["base_model"]
    BM -->|"① save"| DISK["💾 disk<br/>SAVED_MODEL_&lt;prec&gt;"]
    BM -->|"① push"| HUB1["📦 Your Hub<br/>…-from-hub"]
    DISK -->|"③ load"| ML["model_loaded"]
    ML -->|"③ push"| HUB2["📦 Your Hub<br/>…-from-disk"]
    HUB1 -->|"② load + test"| OK2["✅ works"]
    HUB2 -->|"④ load + test"| OK4["✅ works"]
    style DISK fill:#2e7d32,color:#fff,stroke:#1b5e20
```


### Load the model

In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load back the on-disk copy; its saved config carries the precision, so no extra kwargs needed.
model_loaded = AutoModelForCausalLM.from_pretrained(SAVED_DIR, device_map="auto")
tokenizer_loaded = AutoTokenizer.from_pretrained(SAVED_DIR)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Log model and tokenizer

In [21]:
describe(model_loaded, tokenizer_loaded)


--- Model ---
Type: <class 'transformers.models.mistral.modeling_mistral.MistralForCausalLM'>
Architecture: MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((40

### Test the model

#### via Pipeline

In [22]:
test_pipeline(model_loaded, tokenizer_loaded)


AutoGen is a programming technique or a tool that automatically generates source code from high-level specifications. It's often used in software development to help create, maintain, and evolve complex software systems more efficiently.

The high-level specifications can be defined using templates, which describe the structure and behavior of the generated code. These templates can be written in a language suitable for the developer, such as a domain-specific language (DSL).

By using AutoGen, developers can focus on the logic and structure of their code without worrying about the details of implementation, which can lead to increased productivity and fewer bugs. It's a powerful approach for generating code in a variety of domains, including parsers, compilers, runtime systems, and network protocol implementations.


#### via Model and Tokenizer

In [ ]:
test_generate(model_loaded, tokenizer_loaded)


#### via Tool call (single turn)

In [ ]:
test_tool_call(model_loaded, tokenizer_loaded, "What is the current price of MSFT?", STOCK_TOOLS)


#### via Conversation with tool calls

In [23]:
test_tool_conversation(model_loaded, tokenizer_loaded)


NameError: name 'test_tool_conversation' is not defined

### Push my base model from DISK to HUB

In [24]:
# Same transformers 5.x push_to_hub() workaround as the base-model push above.
from huggingface_hub import upload_folder, create_repo

tmp = f"PUSH_TMP_{PRECISION}_from_disk"
create_repo(REPO_FROM_DISK, token=API_TOKEN, exist_ok=True)
model_loaded.save_pretrained(tmp, save_original_format=False)
tokenizer_loaded.save_pretrained(tmp)
upload_folder(repo_id=REPO_FROM_DISK, folder_path=tmp, token=API_TOKEN)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/habert75/my-base-mistral-4bit-from-disk/commit/01bfb8173e6db3a4a9496c5be47c93a6c449bda2', commit_message='Upload folder using huggingface_hub', commit_description='', oid='01bfb8173e6db3a4a9496c5be47c93a6c449bda2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/habert75/my-base-mistral-4bit-from-disk', endpoint='https://huggingface.co', repo_type='model', repo_id='habert75/my-base-mistral-4bit-from-disk'), pr_revision=None, pr_num=None)

# Disk model from HUB

**What this section demonstrates:** the final hop — reload the disk-derived model back from the Hub (`my-base-mistral-<precision>-from-disk`) and test it. Together the four sections prove the *same* model works identically no matter where it comes from: **original Hub → your Hub → local disk → your Hub again.**

```mermaid
flowchart LR
    ORIG["🤗 Original Hub<br/>Mistral-7B-Instruct-v0.3"] -->|"① load"| BM["base_model"]
    BM -->|"① save"| DISK["💾 disk<br/>SAVED_MODEL_&lt;prec&gt;"]
    BM -->|"① push"| HUB1["📦 Your Hub<br/>…-from-hub"]
    DISK -->|"③ load"| ML["model_loaded"]
    ML -->|"③ push"| HUB2["📦 Your Hub<br/>…-from-disk"]
    HUB1 -->|"② load + test"| OK2["✅ works"]
    HUB2 -->|"④ load + test"| OK4["✅ works"]
    style HUB2 fill:#2e7d32,color:#fff,stroke:#1b5e20
    style OK4 fill:#2e7d32,color:#fff,stroke:#1b5e20
```


### Load the model

In [25]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_loaded_fromHF = AutoModelForCausalLM.from_pretrained(
    REPO_FROM_DISK, device_map="auto"
)
tokenizer_loaded_fromHF = AutoTokenizer.from_pretrained(REPO_FROM_DISK)


config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.14GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.67M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.96k [00:00<?, ?B/s]

Log model and tokenizer

In [26]:
describe(model_loaded_fromHF, tokenizer_loaded_fromHF)


--- Model ---
Type: <class 'transformers.models.mistral.modeling_mistral.MistralForCausalLM'>
Architecture: MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((40

### Test the model

#### via Pipeline

In [27]:
test_pipeline(model_loaded_fromHF, tokenizer_loaded_fromHF)


AutoGen (Automatic Generation) is a computer programming technique that automatically generates source code based on a set of specifications or templates. It's a form of metaprogramming, where programs are written about programs.

The process often involves creating a description of the desired functionality in a high-level language, and then using that description as input to a tool that generates the actual source code in a lower-level language, such as C or Java. This can help in reducing the amount of manual coding, increasing productivity, and enhancing code consistency.

AutoGen can be used for various purposes, such as code generation for complex algorithms, database schema generation, and even for generating test cases or documentation. The specifics of how AutoGen is implemented can vary widely, but the goal is always to automate parts of the coding process to make it more efficient and consistent.


#### via Model and Tokenizer

In [ ]:
test_generate(model_loaded_fromHF, tokenizer_loaded_fromHF)


#### via Tool call (single turn)

In [ ]:
test_tool_call(model_loaded_fromHF, tokenizer_loaded_fromHF, "What is the current price of MSFT?", STOCK_TOOLS)


#### via Conversation with tool calls

In [ ]:
test_tool_conversation(model_loaded_fromHF, tokenizer_loaded_fromHF)
